# Entraînement du Modèle MIF - Multi-grained Information Fusion

Ce notebook lance l'entraînement complet du modèle MIF pour la détection de fake news.

## ⚠️ IMPORTANT - Setup Google Drive

**Le dossier `src` se trouve directement dans "Mon Drive" (MyDrive) sur Google Drive.**

### Instructions :
1. **Exécutez la cellule "Setup - Google Drive"** ci-dessous pour monter Drive et copier le dossier `src`
2. Assurez-vous que le dossier `src` est bien dans votre Drive racine (MyDrive)

## Étapes du notebook :
1. Setup - Google Drive (monter Drive et copier src)
2. Installation des dépendances
3. Configuration de l'entraînement
4. Téléchargement/Chargement du dataset
5. Initialisation du modèle et des composants
6. Lancement de l'entraînement
7. Visualisation des résultats

## 0. Setup - Google Drive (IMPORTANT!)

In [ ]:
# Setup Google Drive
# Le dossier src se trouve directement dans "Mon Drive" (MyDrive)

from google.colab import drive
import os

print("🔌 Montage de Google Drive...")
drive.mount('/content/drive')

# Chemin vers le dossier src dans MyDrive
DRIVE_SRC_PATH = '/content/drive/MyDrive/src'

print(f"\n📂 Copie du dossier src depuis Drive...")
print(f"   Chemin source: {DRIVE_SRC_PATH}")

# Vérifier que le dossier existe
if os.path.exists(DRIVE_SRC_PATH):
    # Copier le dossier src
    !cp -r {DRIVE_SRC_PATH} .

    # Vérifier que le dossier a été copié
    print("\n✅ Vérification du dossier src:")
    !ls -la src/
else:
    print(f"⚠️ Dossier non trouvé à: {DRIVE_SRC_PATH}")
    print("\nVérifiez que:")
    print("  1. Le dossier 'src' est bien dans votre Drive racine (MyDrive)")
    print("  2. Vous avez bien monté Google Drive")

## 1. Installation des dépendances

In [ ]:
# Installer les dépendances
!pip install torch transformers torchvision pillow pandas numpy scikit-learn tqdm pytesseract kagglehub

## 2. Imports et Configuration

In [ ]:
import os
import sys
import kagglehub
import torch

# Ajouter src au path
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from src.config import Config

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {Config.DEVICE}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Afficher les informations GPU si disponible
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Configuration de l'entraînement

In [ ]:
# ⚙️ PARAMÈTRES D'ENTRAÎNEMENT - Modifiez selon vos besoins

# Dataset
DATASET_NAME = "gossip"  # "gossip" ou "politi"

# Hyperparamètres (sans GPU / RAM limitée : garder BATCH_SIZE à 2 ou 4)
BATCH_SIZE = 4  # 2 ou 4 si "session crashed after using all available RAM"
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5  # Commencer avec 5, puis augmenter si nécessaire
WEIGHT_DECAY = 0.01

# Options (USE_OCR=False réduit la RAM et accélère si crash mémoire)
USE_OCR = True  # Mettre False si crash RAM (pas d'extraction de texte dans les images)
OUTPUT_DIR = "./outputs"  # Dossier pour sauvegarder les checkpoints

# Sous-ensemble pour entraînement rapide (None = utiliser tout le dataset)
TRAIN_SUBSET_SIZE = 40   # 100 exemples en train (mettre None pour tout utiliser)
TEST_SUBSET_SIZE = 8     # 20 exemples en test (mettre None pour tout utiliser)

# Afficher la configuration
print("📋 Configuration de l'entraînement:")
print(f"   Dataset: {DATASET_NAME}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Use OCR: {USE_OCR}")
print(f"   Train subset: {TRAIN_SUBSET_SIZE if TRAIN_SUBSET_SIZE else 'tout'}")
print(f"   Test subset: {TEST_SUBSET_SIZE if TEST_SUBSET_SIZE else 'tout'}")
print(f"   Output dir: {OUTPUT_DIR}")
print(f"   Device: {Config.DEVICE}")

## 4. Téléchargement/Chargement du Dataset

In [ ]:
# Télécharger ou charger le dataset existant
print("Téléchargement/Chargement du dataset...")
try:
    dataset_path = kagglehub.dataset_download("saikarthikthota/multimodal-fusion-based-fake-news-detection")
    print(f"✅ Dataset trouvé à: {dataset_path}")
except Exception as e:
    print(f"❌ Erreur: {e}")
    print("Veuillez télécharger le dataset manuellement")
    dataset_path = None

In [ ]:
# Vérifier la structure du dataset
if dataset_path:
    train_csv = os.path.join(dataset_path, f"AAAI_dataset/{DATASET_NAME}_train.csv")
    test_csv = os.path.join(dataset_path, f"AAAI_dataset/{DATASET_NAME}_test.csv")
    images_train_dir = os.path.join(dataset_path, f"AAAI_dataset/Images/{DATASET_NAME}_train")
    images_test_dir = os.path.join(dataset_path, f"AAAI_dataset/Images/{DATASET_NAME}_test")

    print(f"Train CSV: {os.path.exists(train_csv)}")
    print(f"Test CSV: {os.path.exists(test_csv)}")
    print(f"Images train dir: {os.path.exists(images_train_dir)}")
    print(f"Images test dir: {os.path.exists(images_test_dir)}")

    if all([os.path.exists(train_csv), os.path.exists(test_csv),
            os.path.exists(images_train_dir), os.path.exists(images_test_dir)]):
        print("\n✅ Tous les fichiers nécessaires sont présents!")
    else:
        print("\n⚠️ Certains fichiers manquent. Vérifiez les chemins.")
else:
    print("❌ Dataset non disponible")

## 5. Initialisation du Modèle et des Composants

In [ ]:
from src.models.mif_model import MIFModel
from src.dataset import FakeNewsDataset
from src.modules.text_encoder import TextEncoder
from src.modules.image_encoder import ImageEncoder
from src.modules.clip_encoder import CLIPEncoder
from torch.utils.data import DataLoader
import pandas as pd

print("Initialisation des encodeurs...")
text_encoder = TextEncoder()
image_encoder = ImageEncoder()
clip_encoder = CLIPEncoder()
print("✅ Encodeurs initialisés")

print("\nCréation du modèle MIF...")
model = MIFModel().to(Config.DEVICE)
print("✅ Modèle créé")

# Compter les paramètres
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n📊 Statistiques du modèle:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

In [ ]:
# Créer les datasets
# Sous-ensemble équilibré (REAL/FAKE) si TRAIN_SUBSET_SIZE / TEST_SUBSET_SIZE sont définis ; sinon dataset complet.
if dataset_path:
    from torch.utils.data import Subset
    from src.dataset import collate_fn

    def normalize_label_series(s):
        """Aligné sur la logique de FakeNewsDataset (labels texte ou numériques)."""
        s2 = s.copy()
        if s2.dtype == object:
            norm = s2.astype(str).str.strip().str.lower()
            mapped = norm.map({"fake": 1, "real": 0})
            numeric = pd.to_numeric(s2, errors="coerce")
            s2 = mapped.fillna(numeric)
        else:
            s2 = pd.to_numeric(s2, errors="coerce")
        return s2

    def write_flipped_label_csv(input_csv_path: str) -> str:
        """Crée un CSV temporaire où label est inversé: 0 <-> 1, dans un dossier writable."""
        df = pd.read_csv(input_csv_path)
        labels = normalize_label_series(df["label"])
        missing_mask = labels.isna()
        if missing_mask.any():
            print(f"⚠️ {missing_mask.sum()} labels non numériques/vides seront conservés tels quels.")
        flipped = labels.copy()
        valid = ~missing_mask
        flipped.loc[valid] = 1 - flipped.loc[valid].astype(int)
        df.loc[valid, "label"] = flipped.loc[valid].astype(int)

        # Kaggle input est read-only: on écrit dans /kaggle/working si disponible
        writable_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
        writable_dir = os.path.join(writable_dir, "flipped_labels")
        os.makedirs(writable_dir, exist_ok=True)

        base_name = os.path.basename(input_csv_path)
        stem, ext = os.path.splitext(base_name)
        flipped_csv_path = os.path.join(writable_dir, f"{stem}_labels_flipped{ext}")
        df.to_csv(flipped_csv_path, index=False)
        return flipped_csv_path

    train_csv = write_flipped_label_csv(train_csv)
    test_csv = write_flipped_label_csv(test_csv)
    print(f"🔁 Labels inversés pour l'entraînement: {train_csv}")
    print(f"🔁 Labels inversés pour le test: {test_csv}")

    def subset_class_counts(subset_size: int):
        """N = n_real + n_fake ; parité (±1 sur REAL si N impair)."""
        half = subset_size // 2
        remainder = subset_size % 2
        n_real = half + remainder
        n_fake = half
        return n_real, n_fake

    def first_k_indices_each_class_in_order(df, k_real, k_fake, label_col="label"):
        labels = normalize_label_series(df[label_col])
        real_idx, fake_idx = [], []
        for pos in range(len(df)):
            val = labels.iloc[pos]
            if pd.isna(val):
                continue
            y = int(val)
            if y == 0 and len(real_idx) < k_real:
                real_idx.append(pos)
            elif y == 1 and len(fake_idx) < k_fake:
                fake_idx.append(pos)
            if len(real_idx) >= k_real and len(fake_idx) >= k_fake:
                break
        if len(real_idx) < k_real or len(fake_idx) < k_fake:
            print("⚠️ Pas assez d'exemples d'une classe dans ce CSV pour atteindre le quota demandé.")
            print(f"   REAL: {len(real_idx)}/{k_real} | FAKE: {len(fake_idx)}/{k_fake}")
        return real_idx + fake_idx

    print("\nCréation des datasets...")
    train_dataset_full = FakeNewsDataset(
        csv_path=train_csv,
        images_dir=images_train_dir,
        text_encoder=text_encoder,
        image_encoder=image_encoder,
        clip_encoder=clip_encoder,
        use_ocr=USE_OCR,
    )

    test_dataset_full = FakeNewsDataset(
        csv_path=test_csv,
        images_dir=images_test_dir,
        text_encoder=text_encoder,
        image_encoder=image_encoder,
        clip_encoder=clip_encoder,
        use_ocr=USE_OCR,
    )

    tr_idx = None
    te_idx = None

    if TRAIN_SUBSET_SIZE is not None:
        n_real_tr, n_fake_tr = subset_class_counts(TRAIN_SUBSET_SIZE)
        df_tr = pd.read_csv(train_csv)
        tr_idx = first_k_indices_each_class_in_order(df_tr, n_real_tr, n_fake_tr)
        train_dataset = Subset(train_dataset_full, tr_idx)
        print(
            f"✅ Train dataset: {len(train_dataset)} échantillons "
            f"(équilibré REAL={n_real_tr}, FAKE={n_fake_tr} ; total CSV train={len(train_dataset_full)})"
        )
    else:
        train_dataset = train_dataset_full
        print(f"✅ Train dataset: {len(train_dataset)} échantillons (complet)")

    if TEST_SUBSET_SIZE is not None:
        n_real_te, n_fake_te = subset_class_counts(TEST_SUBSET_SIZE)
        df_te = pd.read_csv(test_csv)
        te_idx = first_k_indices_each_class_in_order(df_te, n_real_te, n_fake_te)
        test_dataset = Subset(test_dataset_full, te_idx)
        print(
            f"✅ Test dataset: {len(test_dataset)} échantillons "
            f"(équilibré REAL={n_real_te}, FAKE={n_fake_te} ; total CSV test={len(test_dataset_full)})"
        )
    else:
        test_dataset = test_dataset_full
        print(f"✅ Test dataset: {len(test_dataset)} échantillons (complet)")

    print("\n📊 Distribution des labels (train):")
    if TRAIN_SUBSET_SIZE is not None and tr_idx is not None:
        df_train = pd.read_csv(train_csv).iloc[tr_idx]
    else:
        df_train = pd.read_csv(train_csv)
    print(normalize_label_series(df_train["label"]).astype("Int64").value_counts().sort_index())

    print("\n📊 Distribution des labels (test):")
    if TEST_SUBSET_SIZE is not None and te_idx is not None:
        df_test = pd.read_csv(test_csv).iloc[te_idx]
    else:
        df_test = pd.read_csv(test_csv)
    print(normalize_label_series(df_test["label"]).astype("Int64").value_counts().sort_index())

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_fn,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_fn,
    )

    print(f"\n✅ DataLoaders créés")
    print(f"   Train batches: {len(train_loader)}")
    print(f"   Test batches: {len(test_loader)}")
else:
    print("❌ Impossible de créer les datasets sans le dataset_path")


*(La création des datasets et des DataLoaders est entièrement dans la **cellule précédente**. Ne pas dupliquer ce code : une ancienne version commençait par une ligne indentée par erreur, ce qui provoquait `IndentationError`.)*


## 6. Lancement de l'Entraînement

In [ ]:
# Importer les fonctions d'entraînement
from src.train import train_epoch, evaluate
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import numpy as np

# Créer le dossier de sortie
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Loss et optimizer
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print("✅ Optimizer et scheduler initialisés")
print(f"   Loss: CrossEntropyLoss")
print(f"   Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"   Scheduler: CosineAnnealingLR")

In [ ]:
# Boucle d'entraînement
if dataset_path:
    print("\n" + "="*60)
    print("🚀 DÉMARRAGE DE L'ENTRAÎNEMENT")
    print("="*60 + "\n")

    best_f1 = 0
    train_losses = []
    train_accs = []
    test_losses = []
    test_accs = []
    test_f1s = []

    for epoch in range(NUM_EPOCHS):
        print(f"\n{'='*60}")
        print(f"EPOCH {epoch+1}/{NUM_EPOCHS}")
        print(f"{'='*60}")

        # Entraînement
        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, criterion, Config.DEVICE
        )

        # Évaluation
        test_loss, test_acc, test_precision, test_recall, test_f1 = evaluate(
            model, test_loader, criterion, Config.DEVICE
        )

        # Mettre à jour le learning rate
        scheduler.step()

        # Sauvegarder les métriques
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        test_f1s.append(test_f1)

        # Afficher les résultats
        print(f"\n📊 Résultats Epoch {epoch+1}:")
        print(f"   Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
        print(f"   Test  - Loss: {test_loss:.4f}, Acc: {test_acc:.4f}")
        print(f"   Test  - Precision: {test_precision:.4f}, Recall: {test_recall:.4f}, F1: {test_f1:.4f}")
        print(f"   Learning Rate: {scheduler.get_last_lr()[0]:.2e}")

        # Sauvegarder le meilleur modèle (poids seuls : .pt beaucoup plus léger, compatible load_model)
        # Pas d'optimizer/scheduler → impossible de reprendre l'entraînement exactement depuis ce fichier.
        if test_f1 > best_f1:
            best_f1 = test_f1
            checkpoint_path = os.path.join(OUTPUT_DIR, 'best_model.pt')
            tmp_path = checkpoint_path + ".tmp"
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'test_f1': test_f1,
                'test_acc': test_acc,
                'test_precision': test_precision,
                'test_recall': test_recall,
                'config': {
                    'batch_size': BATCH_SIZE,
                    'learning_rate': LEARNING_RATE,
                    'num_epochs': NUM_EPOCHS,
                    'dataset': DATASET_NAME,
                    'use_ocr': USE_OCR
                }
            }, tmp_path)
            os.replace(tmp_path, checkpoint_path)
            print(f"\n💾 Meilleur modèle sauvegardé (poids seuls)! (F1: {test_f1:.4f})")

    print("\n" + "="*60)
    print("✅ ENTRAÎNEMENT TERMINÉ!")
    print("="*60)
    print(f"\n📈 Meilleur F1 score: {best_f1:.4f}")
    print(f"📁 Modèle sauvegardé dans: {OUTPUT_DIR}/best_model.pt")
else:
    print("❌ Impossible de lancer l'entraînement sans le dataset")

## 7. Visualisation des Résultats

In [ ]:
# Visualiser les courbes d'apprentissage
import matplotlib.pyplot as plt

if dataset_path and len(train_losses) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Loss
    axes[0, 0].plot(range(1, NUM_EPOCHS + 1), train_losses, 'b-', label='Train Loss', marker='o')
    axes[0, 0].plot(range(1, NUM_EPOCHS + 1), test_losses, 'r-', label='Test Loss', marker='s')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss au cours de l\'entraînement')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Accuracy
    axes[0, 1].plot(range(1, NUM_EPOCHS + 1), train_accs, 'b-', label='Train Acc', marker='o')
    axes[0, 1].plot(range(1, NUM_EPOCHS + 1), test_accs, 'r-', label='Test Acc', marker='s')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].set_title('Accuracy au cours de l\'entraînement')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # F1 Score
    axes[1, 0].plot(range(1, NUM_EPOCHS + 1), test_f1s, 'g-', label='Test F1', marker='o')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1 Score')
    axes[1, 0].set_title('F1 Score au cours de l\'entraînement')
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    # Résumé des métriques finales
    axes[1, 1].axis('off')
    summary_text = f"""
    📊 RÉSUMÉ FINAL

    Meilleur F1 Score: {max(test_f1s):.4f}
    Meilleure Accuracy: {max(test_accs):.4f}

    Dernière Epoch:
    - Train Loss: {train_losses[-1]:.4f}
    - Train Acc: {train_accs[-1]:.4f}
    - Test Loss: {test_losses[-1]:.4f}
    - Test Acc: {test_accs[-1]:.4f}
    - Test F1: {test_f1s[-1]:.4f}

    Configuration:
    - Dataset: {DATASET_NAME}
    - Batch Size: {BATCH_SIZE}
    - Learning Rate: {LEARNING_RATE}
    - Epochs: {NUM_EPOCHS}
    - Use OCR: {USE_OCR}
    """
    axes[1, 1].text(0.1, 0.5, summary_text, fontsize=12, verticalalignment='center',
                    family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

    print(f"📊 Graphiques sauvegardés dans: {OUTPUT_DIR}/training_curves.png")
else:
    print("Aucune donnée à visualiser")

## 8. Résumé et Prochaines Étapes

In [ ]:
print("=" * 60)
print("✅ ENTRAÎNEMENT COMPLET TERMINÉ!")
print("=" * 60)

if dataset_path:
    print(f"\n📁 Fichiers sauvegardés:")
    print(f"   - Modèle: {OUTPUT_DIR}/best_model.pt")
    print(f"   - Graphiques: {OUTPUT_DIR}/training_curves.png")

    print(f"\n📊 Meilleures performances:")
    if len(test_f1s) > 0:
        best_epoch = np.argmax(test_f1s) + 1
        print(f"   - Meilleur F1: {max(test_f1s):.4f} (Epoch {best_epoch})")
        print(f"   - Meilleure Accuracy: {max(test_accs):.4f} (Epoch {best_epoch})")

    print(f"\n🔜 Prochaines étapes:")
    print(f"   1. Analyser les résultats et les courbes d'apprentissage")
    print(f"   2. Tester le modèle entraîné avec src/inference.py")
    print(f"   3. Implémenter les modules d'explicabilité (LIME/SHAP, Grad-CAM)")
    print(f"   4. Créer l'interface Streamlit")
    print(f"   5. Évaluer sur 50 cas réels (25 vrais, 25 faux)")

    print(f"\n💡 Pour utiliser le modèle entraîné:")
    print(f"   from src.inference import load_model, predict")
    print(f"   model = load_model('{OUTPUT_DIR}/best_model.pt', device)")
else:
    print("❌ L'entraînement n'a pas pu être complété")

In [ ]:
# ===== Choisir une observation =====
SAMPLE_SPLIT = "test"   # "test" ou "train"
SAMPLE_INDEX = 0       # entre 0 et TEST_SUBSET_SIZE-1 si split="test"
IMAGE_FILENAME = None  # optionnel: ex. "some_image.jpg" pour retrouver automatiquement SAMPLE_INDEX

# Chemin vers TON checkpoint dans Drive (à adapter à ton dossier exact)
CHECKPOINT_PATH = "/content/drive/MyDrive/Résultats/best_model_m1_400.pt"

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import torch
import pandas as pd
from torch.utils.data import Subset

from src.inference import load_model, predict
from src.config import Config


def _df_aligned_with_dataset(csv_path, dataset_obj):
    """Même ordre d'exemples que `dataset_obj[i]` (Subset équilibré ou dataset complet)."""
    df_full = pd.read_csv(csv_path)
    if isinstance(dataset_obj, Subset):
        return df_full.iloc[dataset_obj.indices].reset_index(drop=True)
    return df_full


# 1) Charger le bon dataset PyTorch déjà créé dans le notebook
if SAMPLE_SPLIT == "test":
    dataset_obj = test_dataset
    df_sub = _df_aligned_with_dataset(test_csv, test_dataset)
else:
    dataset_obj = train_dataset
    df_sub = _df_aligned_with_dataset(train_csv, train_dataset)

# 2) Option: retrouver SAMPLE_INDEX via le nom de fichier image
if IMAGE_FILENAME is not None:
    matches = df_sub[df_sub["image"] == IMAGE_FILENAME]
    if len(matches) == 0:
        raise ValueError(f"Aucune image trouvée dans le sous-ensemble avec image == {IMAGE_FILENAME}")
    SAMPLE_INDEX = int(matches.index[0])

# 3) Récupérer l'observation exacte
sample = dataset_obj[SAMPLE_INDEX]
text_content = sample["text"]                 # content du CSV (sans OCR)
image_path = sample["image_path"]           # chemin vers l'image sur disque
true_label_int = int(sample["label"].item()) if hasattr(sample["label"], "item") else int(sample["label"])
true_label_str = "REAL" if true_label_int == 0 else "FAKE"

# 4) Visualiser image + texte
img = Image.open(image_path).convert("RGB")
plt.figure(figsize=(10,4))
plt.imshow(img)
plt.axis("off")
plt.show()

print(f"== Observation SPLIT={SAMPLE_SPLIT} index={SAMPLE_INDEX} ==")
print("Label vrai :", true_label_str)
print("\nTexte (content CSV) :\n", text_content)

# 5) Charger le checkpoint best_model_m1_400.pt et mettre en eval
# (si 'model' existe déjà dans le notebook, on recharge le state_dict dans le modèle existant)
if "model" in globals():
    ckpt = torch.load(CHECKPOINT_PATH, map_location=Config.DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    model_for_infer = model
else:
    model_for_infer = load_model(CHECKPOINT_PATH, Config.DEVICE)

# 6) Prédiction sur CETTE observation + confiance
pred_int, confidence, features = predict(
    model_for_infer,
    text_content,   # le predict ajoute l'OCR si USE_OCR=True
    image_path,
    text_encoder,
    image_encoder,
    clip_encoder,
    Config.DEVICE,
    use_ocr=USE_OCR
)
pred_label_str = "REAL" if pred_int == 0 else "FAKE"

print("\n== Résultat modèle ==")
print("Prédiction :", pred_label_str)
print(f"Confiance (proba de la classe prédite) : {confidence:.4f}")
if "clip_similarity" in features:
    print(f"CLIP Similarity : {features['clip_similarity'].item():.4f}")


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

from src.utils.ocr import extract_text_from_pil_image

# Choix du split
DATASET_OBJ = test_dataset   # ou train_dataset

N = 10

for i in range(min(N, len(DATASET_OBJ))):
    sample = DATASET_OBJ[i]
    image_path = sample["image_path"]
    text_content = sample["text"]
    true_int = int(sample["label"].item()) if hasattr(sample["label"], "item") else int(sample["label"])
    true_str = "REAL" if true_int == 0 else "FAKE"

    # Charger l'image puis extraire l'OCR
    img = Image.open(image_path).convert("RGB")
    ocr_text = extract_text_from_pil_image(img)

    print("=" * 80)
    print(f"Index: {i} | Label vrai: {true_str}")
    print("\n--- Texte CSV (content) ---")
    print(text_content)
    print("\n--- OCR extrait (depuis l'image) ---")
    print(ocr_text)

    # Visualisation image
    plt.figure(figsize=(10, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [ ]:
from src.config import Config

# TrOCR (premier appel : téléchargement du modèle Hugging Face, peut prendre du temps)
Config.OCR_BACKEND = "trocr"
# optionnel :
# Config.TROCR_MODEL = "microsoft/trocr-base-printed"

# GPU fortement recommandé pour la boucle sur 10 images
print("OCR backend:", Config.OCR_BACKEND, "| Device:", Config.DEVICE)

In [ ]:
from src.config import Config
Config.OCR_MIN_WORD_CONFIDENCE = 60  # exemple : réduire le bruit

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import torch

from src.utils.ocr import extract_text_from_pil_image
from src.inference import load_model, predict
from src.config import Config

# À adapter
CHECKPOINT_PATH = "/content/drive/MyDrive/Résultats/best_model_m1_400.pt"  # mets ton chemin exact
DATASET_OBJ = test_dataset  # ou train_dataset
N = 10

# En général : tu veux que la prédiction utilise la même option OCR que ton entraînement
PREDICT_USE_OCR = USE_OCR  # si USE_OCR existe dans ton notebook

# Charger le modèle si nécessaire
model_for_infer = load_model(CHECKPOINT_PATH, Config.DEVICE)

for i in range(min(N, len(DATASET_OBJ))):
    sample = DATASET_OBJ[i]

    image_path = sample["image_path"]
    text_content = sample["text"]
    true_int = int(sample["label"].item()) if hasattr(sample["label"], "item") else int(sample["label"])
    true_str = "REAL" if true_int == 0 else "FAKE"

    # OCR extrait (pour inspection)
    img = Image.open(image_path).convert("RGB")
    ocr_text = extract_text_from_pil_image(img)

    # Prédiction modèle
    pred_int, confidence, features = predict(
        model_for_infer,
        text_content,
        image_path,
        text_encoder,
        image_encoder,
        clip_encoder,
        Config.DEVICE,
        use_ocr=PREDICT_USE_OCR
    )
    pred_str = "REAL" if pred_int == 0 else "FAKE"

    print("=" * 100)
    print(f"Index: {i} | Label vrai: {true_str}")
    print(f"Prédiction modèle: {pred_str} | Confiance: {confidence:.4f}")
    if "clip_similarity" in features:
        print(f"CLIP Similarity: {features['clip_similarity'].item():.4f}")

    print("\n--- Texte CSV (content) ---")
    print(text_content)

    print("\n--- OCR extrait (depuis l'image) ---")
    print(ocr_text)

    # Visualisation image
    plt.figure(figsize=(10, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()